In [1]:
%pip install pandas pyarrow textblob textblob-fr

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import json
import os
import glob
from textblob import TextBlob
from textblob_fr import PatternTagger, PatternAnalyzer
from datetime import datetime

BRONZE_DIR = "../bronze"
SILVER_DIR = "."

def process_bronze_to_silver():
    # 1. Trouver le fichier JSON le plus recent dans Bronze
    bronze_files = glob.glob(os.path.join(BRONZE_DIR, "*.json"))
    if not bronze_files:
        print("\u274c Aucun fichier trouve dans la couche Bronze.")
        return None
    
    latest_file = max(bronze_files, key=os.path.getctime)
    print(f"\U0001f4d6 Lecture du fichier brut : {latest_file}")
    
    with open(latest_file, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    
    # 2. Parser les avis
    parsed_reviews = []
    for place in raw_data:
        agency_info = place.get('result', {})
        agency_name = agency_info.get('name', 'Inconnu')
        
        reviews = agency_info.get('reviews', [])
        for review in reviews:
            # Gestion du temps: accepte int (timestamp) ou string
            time_val = review.get('time')
            if isinstance(time_val, (int, float)) and time_val > 0:
                date_review = datetime.fromtimestamp(time_val).strftime('%Y-%m-%d %H:%M:%S')
            elif isinstance(time_val, str):
                date_review = time_val
            else:
                date_review = None
            
            parsed_reviews.append({
                'agency_name': agency_name,
                'author_name': review.get('author_name', 'Anonyme'),
                'rating': review.get('rating'),
                'review_text': review.get('text', ''),
                'review_date': date_review
            })
    
    if not parsed_reviews:
        print("\u274c Aucun avis trouve dans les donnees.")
        return None
    
    df = pd.DataFrame(parsed_reviews)
    
    print(f"\U0001f4ca {len(df)} avis trouves pour {df['agency_name'].nunique()} agences")
    
    # 3. NLP - Analyse de sentiment
    print("\U0001f9e0 Application du modele NLP (TextBlob FR)...")
    
    def get_sentiment_label_fr(text):
        if not text or str(text).strip() == "":
            return 'Neutral'
        blob = TextBlob(text, pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())
        score = blob.sentiment[0] 
        if score >= 0.1:
            return 'Positive'
        elif score <= -0.1:
            return 'Negative'
        else:
            return 'Neutral'

    df['sentiment'] = df['review_text'].apply(get_sentiment_label_fr)
    
    # 4. Nettoyer les anciens fichiers Parquet avant de sauvegarder
    old_files = glob.glob(os.path.join(SILVER_DIR, 'reviews_cleaned_*.parquet'))
    for f in old_files:
        os.remove(f)
        print(f"\U0001f5d1\ufe0f Ancien fichier supprime : {os.path.basename(f)}")
    
    # 5. Sauvegarder le nouveau fichier Parquet
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    silver_filename = f"reviews_cleaned_{timestamp_str}.parquet"
    silver_path = os.path.join(SILVER_DIR, silver_filename)
    
    df.to_parquet(silver_path, engine='pyarrow', index=False)
    
    print(f"\u2705 Transformation Silver terminee !")
    print(f"\U0001f4c1 Fichier Parquet sauvegarde : {silver_path}")
    
    # Afficher un resume par agence
    print(f"\n\U0001f4ca Resume par agence :")
    summary = df.groupby('agency_name').agg(
        nb_avis=('rating', 'count'),
        note_moyenne=('rating', 'mean'),
    ).round(2)
    print(summary.to_string())
    
    return df

# Execution
df_silver = process_bronze_to_silver()

if df_silver is not None:
    display(df_silver.head(10))

📖 Lecture du fichier brut : ../bronze/scraped_reviews_raw_20260406_100039.json
📊 139 avis trouves pour 5 agences
🧠 Application du modele NLP (TextBlob FR)...


🗑️ Ancien fichier supprime : reviews_cleaned_20260406_154129.parquet


✅ Transformation Silver terminee !
📁 Fichier Parquet sauvegarde : ./reviews_cleaned_20260406_155711.parquet

📊 Resume par agence :
                               nb_avis  note_moyenne
agency_name                                         
AL AKHDAR BANK Siège                54          3.69
Attijariwafa Bank                   52          1.90
Banque Populaire                    13          1.54
Banque Populaire Ibn Khaldoun       14          1.36
Crédit Agricole                      6          4.17


,agency_name,author_name,rating,review_text,review_date,sentiment
0,Banque Populaire,Widad Ait ali,1,Une agence catastrophique qui ne sert vraiment...,2025-11-07 10:58:17,Positive
1,Banque Populaire,alex dieu,1,Je suis venu dans cette agence afin d'ouvrir u...,2023-04-07 09:58:17,Positive
2,Banque Populaire,Imane E.,1,J'aurais mis 0 étoile mais ce n'est pas possib...,2023-04-07 09:58:17,Negative
3,Banque Populaire,Nawal Lrhorfi,1,"J’habite à côté de cette agence,elle est juste...",2023-04-07 09:58:17,Negative
4,Banque Populaire,wahib timoulali,1,"Service médiocre , application qui n'est pas m...",2024-04-06 09:58:17,Negative
5,Banque Populaire,SAAD YOUSFI,1,Je n'arrive pas à trouver le numéro de télépho...,2018-04-08 10:58:17,Neutral
6,Banque Populaire,Meryem Itb,1,Le pire service client que j'aie jamais vu ! L...,2026-02-05 10:58:17,Neutral
7,Banque Populaire,August Tempest,4,,2025-09-08 10:58:17,Neutral
8,Banque Populaire,Tarik berrada,1,,2025-04-06 10:58:17,Neutral
9,Banque Populaire,Hamza Lam,1,,2025-04-06 10:58:17,Neutral
